In [114]:
# 09/14
import torch
print(torch.__version__)

2.7.1+cu128


In [83]:
print(torch.cuda.is_available())

True


In [84]:
print(torch.tensor([1,2,3]))
print(torch.Tensor([[1,2,3], [4,5,6]]))
print(torch.LongTensor([1,2,3]))
print(torch.FloatTensor([1,2,3]))

tensor([1, 2, 3])
tensor([[1., 2., 3.],
        [4., 5., 6.]])
tensor([1, 2, 3])
tensor([1., 2., 3.])


In [85]:
tensor = torch.rand(1, 2)
print(tensor.shape)
print(tensor.dtype)
print(tensor.device)

torch.Size([1, 2])
torch.float32
cpu


In [86]:
tensor = torch.rand((3,3), dtype = torch.float)
print(tensor)

tensor([[0.9153, 0.4557, 0.8672],
        [0.4154, 0.8787, 0.6934],
        [0.9962, 0.9869, 0.4152]])


In [87]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [88]:
cpu = torch.FloatTensor([1,2,3])
gpu = torch.cuda.FloatTensor([1,2,3])
tensor = torch.rand((1,1), device=device)
print(cpu)
print(gpu)
print(tensor)

tensor([1., 2., 3.])
tensor([1., 2., 3.], device='cuda:0')
tensor([[0.5444]], device='cuda:0')


In [89]:
cpu = torch.FloatTensor([1,2,3])
gpu = cpu.cuda()
gpu2cpu = gpu.cpu()
cpu2gpu = cpu.to('cuda')

In [90]:
import numpy as np

ndarray = np.array([1,2,3], dtype=np.uint8)
print(torch.tensor(ndarray))
print(torch.Tensor(ndarray))
print(torch.from_numpy(ndarray))

tensor([1, 2, 3], dtype=torch.uint8)
tensor([1., 2., 3.])
tensor([1, 2, 3], dtype=torch.uint8)


In [91]:
tensor = torch.cuda.FloatTensor([1,2,3])
ndarray = tensor.detach().cpu().numpy()
print(ndarray)
print(tensor)

[1. 2. 3.]
tensor([1., 2., 3.], device='cuda:0')


In [92]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [93]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x = df.iloc[:,0].values
        self.y = df.iloc[:,1].values
        self.length = len(df)
    def __getitem__(self, index):
        x = torch.FloatTensor([self.x[index] ** 2, self.x[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y
    def __len__(self):
        return self.length

In [94]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2,1)
    def forward(self, x):
        x = self.layer(x)
        return x

In [95]:
train_dataset = CustomDataset('./non_linear.csv')
train_dataloader = DataLoader(train_dataset,
                             batch_size=128,
                             shuffle=True,
                             drop_last=True)

In [96]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CustomModel().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.0001)

In [97]:
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch +1, cost)

100 tensor(33.7530, device='cuda:0', grad_fn=<DivBackward0>)
200 tensor(10.0253, device='cuda:0', grad_fn=<DivBackward0>)
300 tensor(2.6408, device='cuda:0', grad_fn=<DivBackward0>)
400 tensor(0.6965, device='cuda:0', grad_fn=<DivBackward0>)
500 tensor(0.2462, device='cuda:0', grad_fn=<DivBackward0>)
600 tensor(0.1410, device='cuda:0', grad_fn=<DivBackward0>)
700 tensor(0.0885, device='cuda:0', grad_fn=<DivBackward0>)
800 tensor(0.0808, device='cuda:0', grad_fn=<DivBackward0>)
900 tensor(0.0803, device='cuda:0', grad_fn=<DivBackward0>)
1000 tensor(0.0724, device='cuda:0', grad_fn=<DivBackward0>)


In [98]:
with torch.no_grad():
    model.eval()
    inputs = torch.FloatTensor([[1**2, 1],
                                [5**2, 5],
                                [11*2, 11]]).to(device)
    outputs = model(inputs)
    print(outputs)

tensor([[ 1.9747],
        [69.5369],
        [50.0367]], device='cuda:0')


In [99]:
torch.save(model, './models/model.pt')

In [100]:
torch.save(model.state_dict(), './models/model_state_dict.pt')

In [101]:
dataset = CustomDataset('./non_linear.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset,
                                                        [train_size,
                                                         val_size,
                                                         test_size])
train_dataloader = DataLoader(train_dataset,
                              batch_size=16,
                              shuffle=True,
                              drop_last=True)
val_dataloader = DataLoader(val_dataset,
                              batch_size=4,
                              shuffle=True,
                              drop_last=True)
test_dataloader = DataLoader(test_dataset,
                              batch_size=4,
                              shuffle=False,
                              drop_last=True)

In [102]:
with torch.no_grad():
    model.eval()
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
    outputs = model(inputs)
    print(outputs)

tensor([[ 1.9747],
        [69.5369],
        [50.0367]], device='cuda:0')


In [103]:
torch.save(model, "models/model.pt")

In [104]:
torch.save(model.state_dict(), './models/model_state_dict.pt')

In [ ]:
model = torch.load('./models/model.pt', map_location=device)

In [105]:
print(model)

CustomModel(
  (layer): Linear(in_features=2, out_features=1, bias=True)
)


In [106]:
model_state_dict = torch.load('./models/model_state_dict.pt', map_location=device)
model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [108]:
checkpoint = 1

for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch +1, cost)
        torch.save(model.state_dict(), f'./models/checkpoint-{checkpoint}.pt')
        checkpoint += 1

100 tensor(0.0766, device='cuda:0', grad_fn=<DivBackward0>)
200 tensor(0.0757, device='cuda:0', grad_fn=<DivBackward0>)
300 tensor(0.0769, device='cuda:0', grad_fn=<DivBackward0>)
400 tensor(0.0766, device='cuda:0', grad_fn=<DivBackward0>)
500 tensor(0.0772, device='cuda:0', grad_fn=<DivBackward0>)
600 tensor(0.0765, device='cuda:0', grad_fn=<DivBackward0>)
700 tensor(0.0760, device='cuda:0', grad_fn=<DivBackward0>)
800 tensor(0.0762, device='cuda:0', grad_fn=<DivBackward0>)
900 tensor(0.0773, device='cuda:0', grad_fn=<DivBackward0>)
1000 tensor(0.0755, device='cuda:0', grad_fn=<DivBackward0>)


In [109]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [110]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x1 = df.iloc[:,0].values
        self.x2 = df.iloc[:,1].values
        self.x3 = df.iloc[:,2].values
        self.y = df.iloc[:,3].values
        self.length = len(df)
    def __getitem__(self, index):
        x = torch.FloatTensor([self.x1[index],
                               self.x2[index],
                               self.x3[index]])
        y = torch.FloatTensor([int(self.y[index])])
        return x, y
    def __len__(self):
        return self.length

In [111]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(nn.Linear(3,3),
                                   nn.Sigmoid())

    def forward(self, x):
        x = self.layer(x)
        return x

In [ ]:
dataset = CustomDataset('./binary.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size-train_size-val_size

train_dataset, val_dataset, test_dataset = random_split(dataset,
                                                        [train_size,
                                                         val_size,
                                                         test_size],
                                                         torch.manual_seed(42))
train_dataloader = DataLoader(train_dataset,
                              batch_size=64,
                              shuffle=True,
                              drop_last=True)
val_dataloader = DataLoader(val_dataset,
                              batch_size=4,
                              shuffle=True,
                              drop_last=True)
test_dataloader = DataLoader(test_dataset,
                              batch_size=4,
                              shuffle=False,
                              drop_last=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomModel().to(device)
criterion = nn.BCELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [112]:
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataset:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 1:
        print(epoch, cost)

0 0.0
100 0.0
200 0.0
300 0.0
400 0.0
500 0.0
600 0.0
700 0.0
800 0.0
900 0.0


In [113]:
with torch.no_grad():
    model.eval()
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
        outputs = model(x)
        print(outputs)
        print(outputs >= torch.FloatTensor([0.5]).to(device))

tensor([[185.3714],
        [ 27.7059],
        [ 43.3201],
        [ 94.9626]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[142.5861],
        [  2.1315],
        [258.5341],
        [ 70.9587]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[278.6368],
        [ 11.7730],
        [180.6107],
        [ 58.3014]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[ 30.5716],
        [ 14.9328],
        [  0.2895],
        [162.1880]], device='cuda:0')
tensor([[ True],
        [ True],
        [False],
        [ True]], device='cuda:0')
tensor([[120.4728],
        [296.5446],
        [267.0085],
        [ 81.7498]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')


In [3]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(nn.Linear(1,2),
                                   nn.Sigmoid())
        self.fc = nn.Linear(2,1)
        self._init_weights()
    def _init_weights(self):
        nn.init.xavier_uniform_(self.layer[0].weight)
        self.layer[0].bias.data.fill_(0.01)

        nn.init.xavier_uniform_(self.fc.weight)
        self.fc.bias.data.fill_(0.01)

model = Net()

In [5]:
for name, param in model.named_parameters():
    print(name, param.data)

layer.0.weight tensor([[-0.8305],
        [-0.1674]])
layer.0.bias tensor([0.0100, 0.0100])
fc.weight tensor([[ 1.1511, -0.8173]])
fc.bias tensor([0.0100])


In [10]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layear = nn.Sequential(nn.Linear(1,2),
                                    nn.Sigmoid())
        self.fc = nn.Linear(2,1)
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(self, nn.Linear):
            nn.init.xavier_uniform_(module.weigth)
            nn.init.constant_(module.bias, 0.01)
        print(f'Apply: {module}')

model = Net().to(device)

Apply: Linear(in_features=1, out_features=2, bias=True)
Apply: Sigmoid()
Apply: Sequential(
  (0): Linear(in_features=1, out_features=2, bias=True)
  (1): Sigmoid()
)
Apply: Linear(in_features=2, out_features=1, bias=True)
Apply: Net(
  (layear): Sequential(
    (0): Linear(in_features=1, out_features=2, bias=True)
    (1): Sigmoid()
  )
  (fc): Linear(in_features=2, out_features=1, bias=True)
)


In [11]:
for name, param in model.named_parameters():
    print(name, param.data)

layear.0.weight tensor([[ 0.8477],
        [-0.4246]], device='cuda:0')
layear.0.bias tensor([ 0.9463, -0.3373], device='cuda:0')
fc.weight tensor([[ 0.1665, -0.3802]], device='cuda:0')
fc.bias tensor([-0.3798], device='cuda:0')


In [12]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
x_data = torch.rand(256, 1) * 10
y_data = 2 * x_data + 1 + torch.randn(256, 1) * 0.5

train_dataset = TensorDataset(x_data, y_data)
train_dataloader = DataLoader(train_dataset,
                              batch_size=32,
                              shuffle=True,
                              drop_last=True)
model = nn.Linear(1,1).to(device)
criterion = nn.MSELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [15]:
for epoch in range(10):
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        _lambda = 0.5
        l1_loss = sum(p.abs().sum() for p in model.parameters())

        loss = criterion(output, y) + _lambda * l1_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(epoch, loss.item(), l1_loss.item())

0 1.862362265586853 2.54579758644104
1 2.205580472946167 2.535670042037964
2 1.9457184076309204 2.525961399078369
3 1.8345990180969238 2.516164779663086
4 2.0020086765289307 2.5018742084503174
5 2.017019033432007 2.493699312210083
6 2.1605923175811768 2.4800233840942383
7 1.733595371246338 2.467907428741455
8 1.8510117530822754 2.4568991661071777
9 1.6736773252487183 2.44425106048584


In [16]:
print(model.weight.item(), model.bias.item())

2.1867287158966064 -0.2577067017555237


In [ ]:
# model = nn.Linear(1,1).to(device)
# optimizer = torch.optim.SGD(model.parameters(),
#                             lr=0.01,
#                             weight_decay=0.01)

In [17]:
for epoch in range(20):
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
    print(epoch, loss.item())

0 0.6213288903236389
1 0.7235432863235474
2 0.8304134607315063
3 0.7670324444770813
4 0.5501888990402222
5 0.4986870586872101
6 0.6261500716209412
7 0.5724217295646667
8 0.3197133541107178
9 0.5241587162017822
10 0.596561849117279
11 0.9400710463523865
12 0.7506420612335205
13 0.7018417716026306
14 0.7001948356628418
15 0.4613261818885803
16 0.8273216485977173
17 0.6692862510681152
18 1.0722706317901611
19 0.8273499011993408


In [18]:
import nltk

In [19]:
for resource in ['wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

In [20]:
import nlpaug.augmenter.word as naw

texts = ['Those who can imagine anything, can create the impossible.',
         'We can only see a short distance ahead, but we can see plenty there that needs to be done',
         'If a machine is expected to be infalliable, it cannot bbe intelligent']
aug = naw.ContextualWordEmbsAug(model_path='bert-base-uncased',
                                action='insert',
                                device='cpu')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

c:\Users\KDS10\Documents\KDT17\17-computer_vision\.venv\Lib\site-packages\nlpaug\augmenter\word\context_word_embs.py:123: SyntaxWarning: invalid escape sequence '\s'
  prefix_reg = '(?<=\s|\W)'
c:\Users\KDS10\Documents\KDT17\17-computer_vision\.venv\Lib\site-packages\nlpaug\augmenter\word\context_word_embs.py:124: SyntaxWarning: invalid escape sequence '\s'
  suffix_reg = '(?=\s|\W)'
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

c:\Users\KDS10\Documents\KDT17\17-computer_vision\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Those who can imagine anything, can create the impossible.
those souls who can honestly imagine beyond anything, can create even the impossible.
We can only see a short distance ahead, but we can see plenty there that needs to be done
we can only see a good short distance ahead, but eventually we plenty can see good plenty there without that danger needs to be done
If a machine is expected to be infalliable, it cannot bbe intelligent
but if a mechanical machine is expected never to be infalliable, it definitely cannot need bbe intelligent


In [22]:
import nlpaug.augmenter.char as nac

texts = ['Those who can imagine anything, can create the impossible.',
         'We can only see a short distance ahead, but we can see plenty there that needs to be done',
         'If a machine is expected to be infalliable, it cannot bbe intelligent']
aug = nac.RandomCharAug(action='delete')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Those who can iine anyig, can cate the mpssile.
We can only see a short distance ahead, but we can see plenty there that needs to be done
We can only see a sot distance ahead, but we can see plnt the ha nes to be do
If a machine is expected to be infalliable, it cannot bbe intelligent
If a achi is expet to be inallia, it cannot bbe ntllget


In [24]:
import nlpaug.augmenter.word as naw

texts = ['Those who can imagine anything, can create the impossible.',
         'We can only see a short distance ahead, but we can see plenty there that needs to be done',
         'If a machine is expected to be infalliable, it cannot bbe intelligent']
aug = naw.RandomWordAug(action='swap')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Those who can imagine anything, create the can impossible.
We can only see a short distance ahead, but we can see plenty there that needs to be done
We only see can a short distance ahead, but can we see plenty that there needs to be done
If a machine is expected to be infalliable, it cannot bbe intelligent
A machine if expected is to be infalliable, it cannot intelligent bbe


In [26]:
import nlpaug.augmenter.word as naw

texts = ['Those who can imagine anything, can create the impossible.',
         'We can only see a short distance ahead, but we can see plenty there that needs to be done',
         'If a machine is expected to be infalliable, it cannot bbe intelligent']
aug = naw.SynonymAug(aug_src='wordnet')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Those world health organization rear end imagine anything, bottom create the impossible.
We can only see a short distance ahead, but we can see plenty there that needs to be done
We can only see a inadequate distance forward, but we can see plenty in that location that needs to equal done
If a machine is expected to be infalliable, it cannot bbe intelligent
If a machine make up expected to be infalliable, information technology cannot bbe intelligent


In [30]:
import nlpaug.augmenter.word as naw

texts = ['Those who can imagine anything, can create the impossible.',
         'We can only see a short distance ahead, but we can see plenty there that needs to be done',
         'If a machine is expected to be infalliable, it cannot bbe intelligent']

reversed_tokens = ["can", "can't", "cannot", "could"]
reversed_aug = naw.ReservedAug(reserved_tokens=reversed_tokens)
augmented_texts = reversed_aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Those who can imagine anything, can create the impossible.
We can only see a short distance ahead, but we can see plenty there that needs to be done
We can only see t short distance ahead, but we can see plenty there that needs to be done
If a machine is expected to be infalliable, it cannot bbe intelligent
If n machine is expected to be infalliable, it cannot bbe intelligent


In [31]:
import nlpaug.augmenter.word as naw

texts = ['Those who can imagine anything, can create the impossible.',
         'We can only see a short distance ahead, but we can see plenty there that needs to be done',
         'If a machine is expected to be infalliable, it cannot bbe intelligent']


back_translation = naw.BackTranslationAug(from_model_name='facebook/wmt19-en-de',
                                          to_model_name='facebook/wmt19-de-end',
                                          device='cpu')


config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

c:\Users\KDS10\Documents\KDT17\17-computer_vision\.venv\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KDS10\.cache\huggingface\hub\models--facebook--wmt19-en-de. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

OSError: facebook/wmt19-de-end is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
augmented_texts = back_translation.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)